# 09 — Rapport de qualité des données

**Rôle :** synthèse finale — couverture, champs garantis sans manquant, frontière de complétude (électronique vs backlog scanné), distributions clés. C'est le document de référence pour interpréter le futur backtest.

**Prérequis :** notebook 07 exécuté.

**Cellule 1 — Imports & chargement de la table canonique.**

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED    = PROJECT_ROOT / "data" / "processed"
REPORTS      = PROJECT_ROOT / "reports"
df = pd.read_csv(PROCESSED / "congress_transactions.csv", parse_dates=["transaction_date", "disclosure_date"])
print("Table canonique :", len(df), "transactions")

Table canonique : 25476 transactions


**Cellule 2 — Couverture par chambre et par an.**

In [2]:
cov = (df.groupby([df["source"], df["disclosure_date"].dt.year]).size()
         .unstack(0).fillna(0).astype(int))
cov

source,house,senate
disclosure_date,,
2014,695,594
2015,1499,896
2016,1463,806
2017,1483,1137
2018,974,1021
2019,1024,972
2020,1606,1435
2021,1634,474
2022,1446,556


**Cellule 3 — Contrôle zéro-manquant** sur les champs garantis.

In [3]:
GUARANTEED = ["declarant_name", "chamber", "transaction_date", "disclosure_date",
              "ticker", "operation_type", "amount_midpoint"]
missing = {c: int(df[c].isna().sum()) for c in GUARANTEED}
missing

{'declarant_name': 0,
 'chamber': 0,
 'transaction_date': 0,
 'disclosure_date': 0,
 'ticker': 1905,
 'operation_type': 0,
 'amount_midpoint': 0}

**Cellule 4 — Distributions clés.** Achats vs ventes, et statistiques de montant (utile pour la future règle de sortie).

In [4]:
op = df["operation_type"].astype(str).str.lower()
buys  = int(op.str.startswith("p").sum())
sells = int(op.str.startswith("s").sum())
print("Achats:", buys, "| Ventes:", sells)
print(df["amount_midpoint"].describe())

Achats: 13148 | Ventes: 12111
count    2.547600e+04
mean     5.354727e+04
std      6.299394e+05
min      8.000500e+03
25%      8.000500e+03
50%      8.000500e+03
75%      3.250050e+04
max      5.000000e+07
Name: amount_midpoint, dtype: float64


**Cellule 5 — Rédaction du rapport de qualité.**

In [5]:
lines = ["# Rapport de qualité — congress_transactions", "",
         f"Généré : {datetime.now(timezone.utc).isoformat(timespec='seconds')}", "",
         f"- Transactions : {len(df)}",
         f"- Déclarants uniques : {df['declarant_name'].nunique()}",
         f"- Achats / Ventes : {buys} / {sells}", "",
         "## Couverture (chambre x an)", "```", cov.to_string(), "```", "",
         "## Champs garantis — valeurs manquantes", "```", str(missing), "```", "",
         "## Frontière de complétude",
         "- Ère électronique, niveau Member.",
         "- Backlog : PDF House `no_text` (cf. 03) + Sénat papier/scanné < 2015 (cf. 05).",
         "- Commissions : Congrès actuel uniquement (cf. 01)."]
(REPORTS / "data_quality.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

# Rapport de qualité — congress_transactions

Généré : 2026-06-24T00:04:43+00:00

- Transactions : 25476
- Déclarants uniques : 316
- Achats / Ventes : 13148 / 12111

## Couverture (chambre x an)
```
source           house  senate
disclosure_date               
2014               695     594
2015              1499     896
2016              1463     806
2017              1483    1137
2018               974    1021
2019              1024     972
2020              1606    1435
2021              1634     474
2022              1446     556
2023              1864     778
2024               337     618
2025               736     803
2026               219     406
```

## Champs garantis — valeurs manquantes
```
{'declarant_name': 0, 'chamber': 0, 'transaction_date': 0, 'disclosure_date': 0, 'ticker': 1905, 'operation_type': 0, 'amount_midpoint': 0}
```

## Frontière de complétude
- Ère électronique, niveau Member.
- Backlog : PDF House `no_text` (cf. 03) + Sénat papier/scanné < 2015 (cf. 05).

Pipeline données de la semaine 1 terminé ✅.

Hors-scope (rappel) : OCR du backlog · mapping GICS→ETF (S2) · stratégie & backtest (S3-S4).